In [4]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from nrgpt_analysis import (load_nrgpt, load_gpt2, make_context, per_word_energies, per_word_surprisal,
                             per_word_surprisal_gpt2,
                             per_word_predictive_energies,
                             per_word_predictive_conditional_energies,
                             plot_energy_landscape_2d, plot_energy_landscape_pca)

MODEL = "nrgpt_local"
MODEL = "../nrgpt/out-OWT02_owt_best_configs/Best_OWT02_owt_best_configs_model=NRGPT_H_FF2W_embed=1536_depth=6_heads=12_LR=3e-05_minLR=None_minLrDiv=10.0_numIter=100000_exp_kko52p3j.pt"

model, tokenizer = load_nrgpt(MODEL)
ctx = make_context(model, tokenizer)
print(f"Block: {ctx.block.__class__.__name__}  |  block_size={ctx.block_size}")

Model loaded from ../nrgpt/out-OWT02_owt_best_configs/Best_OWT02_owt_best_configs_model=NRGPT_H_FF2W_embed=1536_depth=6_heads=12_LR=3e-05_minLR=None_minLrDiv=10.0_numIter=100000_exp_kko52p3j.pt
Block: BlockGrad_FF2W  |  block_size=1024


In [5]:
# getting gpt2 small

modelgpt2, tokenizergpt2 = load_gpt2("gpt2")


In [6]:
enc = tokenizergpt2("In recent years, researchers have discovered that ", return_tensors="pt")
with torch.no_grad():
      output = modelgpt2.generate(
      enc.input_ids,
      attention_mask=enc.attention_mask,
      pad_token_id=tokenizergpt2.eos_token_id,
      max_new_tokens=20,
  )
print(tokenizergpt2.decode(output[0]))

In recent years, researchers have discovered that erythrocytes are the most abundant of all living cells in the body. They are the most


In [7]:
stimuli = pd.read_csv("psych_data/frank_etal/stimuli.txt", sep="\t", encoding="cp1252")
print("Stimuli shape:", stimuli.shape)
print(stimuli.head())

Stimuli shape: (361, 4)
   sent_nr                        sentence               question answer
0        1  Anne lost control and laughed.                      -      -
1        2    Billy wrote on the envelope.  Was it Bob who wrote?      n
2        3    He called over his shoulder.                      -      -
3        4     He stayed against the wall.                      -      -
4        5        Helen ran to the toilet.  Did Helen go quickly?      y


In [8]:
all_word_rows = []
for _, row in stimuli.iterrows():
    sent_nr  = row["sent_nr"]
    words    = str(row["sentence"]).split()

    records_c, _      = per_word_predictive_conditional_energies(words, ctx)
    records_e, ranges = per_word_energies(words, ctx)
    surprisals, _     = per_word_surprisal(words, ctx, n_steps=6)
    surprisalsgpt2, _ = per_word_surprisal_gpt2(words, modelgpt2, tokenizergpt2)
    records_p, _      = per_word_predictive_energies(words, ctx)

    print(f"sent {sent_nr:3d}: {len(words)} words, {len(ranges)} processed")
    for i, (rec_e, surp, surpgpt2, rec_p, rec_c) in enumerate(
            zip(records_e, surprisals, surprisalsgpt2, records_p, records_c)):
        all_word_rows.append({
            "sent_nr":      sent_nr,
            "word_pos":     i + 1,
            **rec_e, **rec_p, **rec_c,
            "surprisal":    surp,
            "surprisalgpt2": surpgpt2,
        })

energies = pd.DataFrame(all_word_rows)
print(f"\nenergies shape: {energies.shape}")
print(energies.head())

sent   1: 5 words, 5 processed
sent   2: 5 words, 5 processed
sent   3: 5 words, 5 processed
sent   4: 5 words, 5 processed
sent   5: 5 words, 5 processed
sent   6: 5 words, 5 processed
sent   7: 5 words, 5 processed
sent   8: 5 words, 5 processed
sent   9: 5 words, 5 processed
sent  10: 5 words, 5 processed
sent  11: 5 words, 5 processed
sent  12: 5 words, 5 processed
sent  13: 6 words, 6 processed
sent  14: 6 words, 6 processed
sent  15: 6 words, 6 processed
sent  16: 6 words, 6 processed
sent  17: 6 words, 6 processed
sent  18: 6 words, 6 processed
sent  19: 6 words, 6 processed
sent  20: 6 words, 6 processed
sent  21: 6 words, 6 processed
sent  22: 6 words, 6 processed
sent  23: 6 words, 6 processed
sent  24: 6 words, 6 processed
sent  25: 6 words, 6 processed
sent  26: 6 words, 6 processed
sent  27: 6 words, 6 processed
sent  28: 6 words, 6 processed
sent  29: 6 words, 6 processed
sent  30: 6 words, 6 processed
sent  31: 6 words, 6 processed
sent  32: 6 words, 6 processed
sent  33

In [9]:
energies.to_csv("psych_data/frank_etal/ucl_with_energy_surprisal_measures.csv", index=False)

In [10]:
et  = pd.read_csv("psych_data/frank_etal/eyetracking.RT.with_ngrams.csv",       sep="\t", encoding="cp1252")
spr = pd.read_csv("psych_data/frank_etal/selfpacedreading.RT.with_ngrams.csv",  sep="\t", encoding="cp1252")

et_full  = energies.merge(et,  on=["sent_nr", "word_pos"], how="inner")
spr_full = energies.merge(spr, on=["sent_nr", "word_pos"], how="inner")

print(f"ET  full shape : {et_full.shape}")
print(f"SPR full shape : {spr_full.shape}")
print(et_full.head())
print(spr_full.head())

ET  full shape : (81109, 154)
SPR full shape : (353584, 151)
   sent_nr  word_pos word_x  n_bpe  start   E_attn_0      E_ff_0   E_total_0  \
0        1         1   Anne      1      0 -54.117828 -394.922729 -449.040558   
1        1         1   Anne      1      0 -54.117828 -394.922729 -449.040558   
2        1         1   Anne      1      0 -54.117828 -394.922729 -449.040558   
3        1         1   Anne      1      0 -54.117828 -394.922729 -449.040558   
4        1         1   Anne      1      0 -54.117828 -394.922729 -449.040558   

     E_attn_1      E_ff_1  ...  word_y  RTfirstfix  RTfirstpass  RTrightbound  \
0  940.437317  806.453857  ...    Anne         216          348           348   
1  940.437317  806.453857  ...    Anne         228          228           228   
2  940.437317  806.453857  ...    Anne         348          348           348   
3  940.437317  806.453857  ...    Anne          68          300           300   
4  940.437317  806.453857  ...    Anne           0   

In [11]:
print(et.shape)
print(et_full.shape)

(81109, 16)
(81109, 154)


In [12]:
et_full.to_csv("ucl_et_full_100.csv",   index=False)
spr_full.to_csv("ucl_spr_full_100.csv", index=False)